# NB1: *Profiling*, medir antes de optimizar

**Computación de Altas Prestaciones para Ciencia de Datos (CAPCD)**

---

## Objetivos de este notebook

1. Usar `%%time` y `%%timeit` para medir tiempos rápidamente.
2. Identificar funciones lentas con `cProfile`.
3. Hacer profiling línea a línea con `line_profiler`.
4. Detectar consumo excesivo de memoria con `memory_profiler`.

**Tiempo estimado:** 2 horas

In [ ]:
# Instalación de dependencias
!pip install -q line_profiler memory_profiler

---

## 1. Mediciones rápidas con magic commands

Jupyter/Colab proporciona dos comandos "mágicos" para medir tiempos:

| Comando | Qué hace | Cuándo usarlo |
|---|---|---|
| `%time` | Mide **una** ejecución | Operaciones lentas (>1s) |
| `%timeit` | Ejecuta múltiples veces y da estadísticas | Operaciones rápidas (<1s) |
| `%%time` | Mide toda la celda | Celdas con varias líneas |
| `%%timeit` | Mide toda la celda (multiple runs) | Benchmarks rigurosos |

**Diferencia clave:** `%` actúa sobre una línea, `%%` sobre toda la celda.

## Demo 1: `%time` vs `%timeit`

In [ ]:
import numpy as np

data = np.random.rand(1_000_000)

In [ ]:
# %time: una sola ejecución
%time resultado = np.sort(data.copy())

In [ ]:
# %timeit: múltiples ejecuciones para estadísticas fiables
%timeit np.sort(data.copy())

In [ ]:
# %%time con celda completa
%%time
# Simular un pipeline de procesamiento
data_copy = data.copy()
data_sorted = np.sort(data_copy)
media = np.mean(data_sorted)
std = np.std(data_sorted)

---

## 2. Profiling de funciones con `cProfile`

`cProfile` es un profiler incluido en la librería estándar de Python. Nos dice en qué funciones se gasta el tiempo.

Es útil como primer paso: identifica las funciones candidatas a optimizar. Después, usamos `line_profiler` para analizar exactamente qué líneas son las lentas.

### Columnas importantes de cProfile:

| Columna | Significado |
|---|---|
| `ncalls` | Nº veces que se llamó a la función |
| `tottime` | Tiempo **total** dentro de la función (sin subfunciones) |
| `cumtime` | Tiempo **acumulado** (incluye subfunciones) |
| `percall` | Tiempo por llamada |

## Demo 2: Profiling con cProfile

In [ ]:
import cProfile
import numpy as np
from io import StringIO
import pstats


def generar_datos(n):
    """Genera datos aleatorios."""
    return np.random.rand(n)


def procesar_datos(data):
    """Realiza operaciones costosas sobre los datos."""
    # Operación 1: ordenar
    sorted_data = np.sort(data)
    # Operación 2: calcular estadísticas
    stats = calcular_estadisticas(sorted_data)
    # Operación 3: un bucle lento en Python puro (cuello de botella)
    resultado = bucle_lento(sorted_data)
    return resultado


def calcular_estadisticas(data):
    """Calcula estadísticas básicas."""
    return {
        "media": np.mean(data),
        "std": np.std(data),
        "mediana": np.median(data),
    }


def bucle_lento(data):
    """Un bucle intencionalmente lento en Python puro."""
    total = 0.0
    for i in range(len(data)):
        total += data[i] ** 2
    return total


def pipeline():
    data = generar_datos(500_000)
    resultado = procesar_datos(data)
    return resultado


# Ejecutar profiling
profiler = cProfile.Profile()
profiler.enable()
resultado = pipeline()
profiler.disable()

# Mostrar resultados ordenados por tiempo acumulado
stats = pstats.Stats(profiler)
stats.sort_stats("cumulative")
stats.print_stats(15)

### Interpretación

Observa que `bucle_lento` consume la **gran mayoría** del tiempo. Esa es la función que deberíamos optimizar primero (en los notebooks 3 y 4 veremos cómo).

`calcular_estadisticas` y `np.sort` son rápidas porque NumPy las ejecuta en C internamente.

---

## 3. Profiling línea a línea con `line_profiler`

`cProfile` nos dice *qué función* es lenta. `line_profiler` nos dice *qué línea exacta* dentro de esa función es la responsable.

En Jupyter/Colab usamos el magic `%lprun`.

## Demo 3: `line_profiler` en acción

In [ ]:
%load_ext line_profiler

In [ ]:
def analisis_completo(n):
    """Función con varias operaciones de distinto coste."""
    # Generar datos
    data = np.random.rand(n)

    # Operación 1: ordenar (numpy, rápido)
    sorted_data = np.sort(data)

    # Operación 2: buscar duplicados (Python puro, lento)
    unicos = set()
    for val in data:
        unicos.add(round(val, 4))

    # Operación 3: cálculo vectorizado (numpy, rápido)
    resultado = np.sum(np.exp(sorted_data[:1000]))

    # Operación 4: concatenación de strings (Python, lento)
    report = ""
    for i in range(10_000):
        report += f"item_{i},"

    return resultado

In [ ]:
%lprun -f analisis_completo analisis_completo(500_000)

### Interpretación

La columna `% Time` revela exactamente qué líneas consumen más tiempo. Normalmente:
- Los **bucles en Python puro** (set, concatenación de strings) dominan.
- Las **operaciones NumPy** (`np.sort`, `np.sum`, `np.exp`) son muy rápidas.

Con esta información, sabemos exactamente qué optimizar.

---

## 4. Profiling de memoria con `memory_profiler`

No solo importa la velocidad. Si tu programa consume más memoria de la disponible, vuestra máquina matará el proceso.

`memory_profiler` nos muestra el consumo de memoria línea a línea.

En Jupyter/Colab usamos el magic `%memit` (como `%timeit` pero para memoria) y `%mprun` (como `%lprun` pero para memoria).

## Demo 4: Profiling de memoria

In [ ]:
%load_ext memory_profiler

In [ ]:
# %memit: medir uso de memoria de una expresión
%memit np.zeros(10_000_000)  # 80 MB (float64 × 10M)

In [ ]:
%memit np.zeros(10_000_000, dtype=np.float32)  # 40 MB (float32 × 10M)

In [ ]:
%%writefile _demo_memoria.py
# Escribimos a archivo porque %mprun necesita un módulo importable
import numpy as np

def crear_datos_ineficiente(n):
    """Ejemplo de uso ineficiente de memoria."""
    # Crea una lista de Python (mucha memoria por el overhead de objetos)
    lista = [float(i) for i in range(n)]
    # La convierte a numpy (duplica la memoria temporalmente)
    array = np.array(lista)
    # Calcula resultado
    resultado = np.sum(array ** 2)
    return resultado

def crear_datos_eficiente(n):
    """Ejemplo de uso eficiente de memoria."""
    # Crea directamente en numpy (sin lista intermedia)
    array = np.arange(n, dtype=np.float64)
    resultado = np.sum(array ** 2)
    return resultado

In [ ]:
from _demo_memoria import crear_datos_ineficiente, crear_datos_eficiente

print("=== Versión INEFICIENTE ===")
%mprun -f crear_datos_ineficiente crear_datos_ineficiente(2_000_000)

In [ ]:
print("=== Versión EFICIENTE ===")
%mprun -f crear_datos_eficiente crear_datos_eficiente(2_000_000)

### Conclusión

La versión ineficiente usa ~3x más memoria porque:
1. La lista de Python tiene overhead por cada objeto `float`.
2. `np.array(lista)` crea una copia, duplicando la memoria temporalmente.

**Consejo:** Siempre que puedas, crea arrays NumPy directamente en lugar de convertir listas.

---

## Ejercicio 1: Encuentra el cuello de botella

La siguiente función procesa datos simulados de sensores. Usa `cProfile` para identificar cuál de las subfunciones es la más lenta.

In [ ]:
import numpy as np


def leer_sensores(n):
    """Simula la lectura de datos de sensores."""
    return np.random.rand(n) * 100  # temperaturas entre 0 y 100


def filtrar_anomalias(data, umbral=95):
    """Filtra valores por encima del umbral (Python puro, lento)."""
    resultado = []
    for val in data:
        if val > umbral:
            resultado.append(val)
    return resultado


def calcular_media_movil(data, ventana=100):
    """Calcula media móvil (Python puro, muy lento)."""
    resultado = []
    for i in range(ventana, len(data)):
        media = sum(data[i - ventana:i]) / ventana
        resultado.append(media)
    return resultado


def generar_reporte(anomalias):
    """Genera un texto de reporte."""
    lineas = []
    for i, val in enumerate(anomalias):
        lineas.append(f"Alerta #{i}: temp={val:.2f}")
    return "\n".join(lineas)


def pipeline_sensores():
    data = leer_sensores(100_000)
    anomalias = filtrar_anomalias(data)
    media_movil = calcular_media_movil(data, ventana=50)
    reporte = generar_reporte(anomalias)
    return reporte

In [ ]:
# TODO: Usa cProfile para ejecutar pipeline_sensores() y encontrar la función más lenta.
# Pista: usa cProfile.Profile(), .enable(), .disable() y pstats.Stats()
# Ordena por 'cumulative' o 'tottime'

import cProfile
import pstats

# TODO: Escribe tu código aquí

In [ ]:
# ✅ Pregunta de comprensión:
# ¿Cuál es la función más lenta del pipeline?
funcion_mas_lenta = ""  # TODO: escribe el nombre de la función (como string)

assert funcion_mas_lenta == "calcular_media_movil", "Revisa los resultados de cProfile"
print("✅ ¡Correcto! calcular_media_movil es el cuello de botella.")

---

## Ejercicio 2: Profiling línea a línea

Ahora que sabemos que `calcular_media_movil` es la función lenta, usa `%lprun` para ver exactamente **qué línea** dentro de ella consume más tiempo.

In [ ]:
data = leer_sensores(50_000)

# TODO: Usa %lprun para hacer profiling de calcular_media_movil
# Pista: %lprun -f nombre_funcion nombre_funcion(argumentos)

**Pregunta:** ¿Qué línea consume más tiempo? ¿Por qué crees que es así?

---

## Ejercicio 3: Optimiza y mide

Reemplaza la función `calcular_media_movil` por una versión que use `np.convolve` (vectorizada). Mide ambas versiones con `%timeit` y calcula el speedup.

In [ ]:
def calcular_media_movil_numpy(data, ventana=100):
    """Calcula media móvil usando NumPy (vectorizado)."""
    # TODO: Usa np.convolve con un kernel de unos/ventana en modo 'valid'
    # Pista: kernel = np.ones(ventana) / ventana
    pass


data = np.random.rand(50_000)

In [ ]:
print("Versión Python puro:")
%timeit calcular_media_movil(data, ventana=50)

print("\nVersión NumPy:")
%timeit calcular_media_movil_numpy(data, ventana=50)

In [ ]:
# ✅ Autoevaluación
res_python = calcular_media_movil(data, ventana=50)
res_numpy = calcular_media_movil_numpy(data, ventana=50)

assert res_numpy is not None, "La función numpy no debe devolver None"
assert len(res_numpy) == len(res_python), f"Longitudes distintas: {len(res_numpy)} vs {len(res_python)}"
assert np.allclose(res_numpy, res_python, atol=1e-10), "Los resultados no coinciden"
print("✅ ¡Correcto! Ambas versiones dan el mismo resultado, pero NumPy es mucho más rápida.")

---

## Ejercicio 4: Memoria

Usa `%memit` para comparar el uso de memoria de crear una matriz de 5 millones de elementos con:
- `dtype=np.float64` (por defecto)
- `dtype=np.float32`
- `dtype=np.int16`

¿Cuánta memoria ahorra usar un dtype más pequeño?

In [ ]:
N = 5_000_000

# TODO: Usa %memit para medir cada una
# %memit np.zeros(N, dtype=...)

---

## Alternativa moderna: Scalene

[**Scalene**](https://github.com/plasma-umass/scalene) es un profiler de última generación que combina **CPU + memoria + GPU** en una sola herramienta, con mucho menos overhead que `line_profiler` + `memory_profiler` por separado.

**Ventajas frente a lo que hemos visto:**
- **Un solo profiler** para CPU, memoria y GPU (en vez de 3 herramientas separadas).
- **Overhead mínimo** (~20%), frente al overhead significativo de `memory_profiler`.
- Distingue entre tiempo en **Python** vs tiempo en **código nativo** (NumPy/C) vs **sistema** (I/O).
- No requiere decoradores `@profile`.

**Cómo usarlo en local**:

```bash
pip install scalene
python -m scalene mi_script.py
```

O desde un notebook local con Jupyter:
```python
%load_ext scalene
%%scalene
# ... tu código aquí ...
```

---

## Resumen

| Herramienta | ¿Qué mide? | Cuándo usarla |
|---|---|---|
| `%time` / `%%time` | Tiempo (1 ejecución) | Mediciones rápidas |
| `%timeit` / `%%timeit` | Tiempo (estadísticas) | Benchmarks rigurosos |
| `cProfile` | Tiempo por función | Encontrar la función lenta |
| `%lprun` (`line_profiler`) | Tiempo por línea | Saber *qué línea* optimizar |
| `%memit` / `%mprun` (`memory_profiler`) | Memoria | Detectar fugas o excesos |

### Flujo de trabajo recomendado

1. **`cProfile`** → ¿Qué función es lenta?
2. **`line_profiler`** → ¿Qué línea dentro de esa función?
3. **Optimizar** → Vectorizar, compilar (Numba), o mover a GPU
4. **Medir de nuevo** → ¿Mejoró? ¿Cuánto?

En el próximo notebook veremos la primera estrategia de optimización: **paralelismo en CPU**.